In [0]:
# ============================================
# 01. RUN SETUP NOTEBOOK
# ============================================

%run ./00_Setup.ipynb

#%ls ls '/Workspace/Users/nirmalworks93@gmail.com/Bookstore_Medallion_Project/'

In [0]:
# ============================================
# 02. DEFINE TABLE NAMES
# ============================================

BRONZE_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"

# Bronze tables
BRONZE_CUSTOMERS = f"{BRONZE_SCHEMA}.customers"
BRONZE_BOOKS = f"{BRONZE_SCHEMA}.books"
BRONZE_ORDERS = f"{BRONZE_SCHEMA}.orders"

# Silver tables
SILVER_CUSTOMERS = f"{SILVER_SCHEMA}.customers"
SILVER_BOOKS = f"{SILVER_SCHEMA}.books"
SILVER_ORDERS = f"{SILVER_SCHEMA}.orders"
SILVER_ORDER_DETAILS = f"{SILVER_SCHEMA}.order_details"

print("Bronze schema:", BRONZE_SCHEMA)
print("Silver schema:", SILVER_SCHEMA)

print("\nBronze tables:")
print(BRONZE_CUSTOMERS)
print(BRONZE_BOOKS)
print(BRONZE_ORDERS)

print("\nSilver tables:")
print(SILVER_CUSTOMERS)
print(SILVER_BOOKS)
print(SILVER_ORDERS)
print(SILVER_ORDER_DETAILS)

In [0]:
# ============================================
# 03. CREATE SILVER SCHEMA
# ============================================

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}
""")

print("Silver schema is ready.")

In [0]:
# ============================================
# 04. READ BRONZE TABLES
# ============================================

bronze_customers_df = spark.table(BRONZE_CUSTOMERS)

bronze_books_df = spark.table(BRONZE_BOOKS)

bronze_orders_df = spark.table(BRONZE_ORDERS)

print("Bronze Customers:", bronze_customers_df.count())
print("Bronze Books    :", bronze_books_df.count())
print("Bronze Orders   :", bronze_orders_df.count())

In [0]:
# ============================================
# 05. INSPECT BRONZE DATA
# ============================================

print("BRONZE CUSTOMERS")
bronze_customers_df.printSchema()
display(bronze_customers_df.limit(10))

print("BRONZE BOOKS")
bronze_books_df.printSchema()
display(bronze_books_df.limit(10))

print("BRONZE ORDERS")
bronze_orders_df.printSchema()
display(bronze_orders_df.limit(10))

In [0]:
# ============================================
# 06. CLEAN CUSTOMERS
# ============================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_customers_df = (
    bronze_customers_df

    # Remove duplicate customer records
    .dropDuplicates(["customer_id"])

    # Remove records without a customer ID
    .filter(
        F.col("customer_id").isNotNull()
    )

    # Clean string columns
    .withColumn(
        "first_name",
        F.initcap(F.trim(F.col("first_name")))
    )
    .withColumn(
        "last_name",
        F.initcap(F.trim(F.col("last_name")))
    )
    .withColumn(
        "email",
        F.lower(F.trim(F.col("email")))
    )
    .withColumn(
        "city",
        F.initcap(F.trim(F.col("city")))
    )

    # Convert timestamp
    .withColumn(
        "created_at",
        F.to_timestamp(F.col("created_at"))
    )

    # Remove records with invalid required fields
    .filter(
        F.col("first_name").isNotNull()
        & F.col("last_name").isNotNull()
        & F.col("email").isNotNull()
        & F.col("city").isNotNull()
    )

    # Add processing timestamp
    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

print("Cleaned customer records:", silver_customers_df.count())

display(silver_customers_df.limit(10))

In [0]:
# ============================================
# 07. CLEAN BOOKS
# ============================================

silver_books_df = (
    bronze_books_df

    # Remove duplicate book records
    .dropDuplicates(["book_id"])

    # Remove records without a book ID
    .filter(
        F.col("book_id").isNotNull()
    )

    # Standardize text columns
    .withColumn(
        "title",
        F.trim(F.col("title"))
    )
    .withColumn(
        "category",
        F.initcap(F.trim(F.col("category")))
    )

    # Convert price to a numeric type
    .withColumn(
        "price",
        F.col("price").cast("decimal(10,2)")
    )

    # Convert timestamp
    .withColumn(
        "created_at",
        F.to_timestamp(F.col("created_at"))
    )

    # Validate required fields and price
    .filter(
        F.col("title").isNotNull()
        & F.col("category").isNotNull()
        & F.col("price").isNotNull()
        & (F.col("price") > 0)
    )

    # Add processing timestamp
    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

print("Cleaned book records:", silver_books_df.count())

display(silver_books_df.limit(10))

In [0]:
# ============================================
# 08. CLEAN ORDERS
# ============================================

silver_orders_df = (
    bronze_orders_df

    # Remove duplicate order IDs
    .dropDuplicates(["order_id"])

    # Remove records without an order ID
    .filter(
        F.col("order_id").isNotNull()
    )

    # Cast numeric fields
    .withColumn(
        "quantity",
        F.col("quantity").cast("int")
    )
    .withColumn(
        "unit_price",
        F.col("unit_price").cast("decimal(10,2)")
    )
    .withColumn(
        "total_amount",
        F.col("total_amount").cast("decimal(12,2)")
    )

    # Standardize status
    .withColumn(
        "order_status",
        F.upper(F.trim(F.col("order_status")))
    )

    # Convert timestamp
    .withColumn(
        "order_timestamp",
        F.to_timestamp(F.col("order_timestamp"))
    )

    # Validate order fields
    .filter(
        F.col("customer_id").isNotNull()
        & F.col("book_id").isNotNull()
        & F.col("quantity").isNotNull()
        & (F.col("quantity") > 0)
        & F.col("unit_price").isNotNull()
        & (F.col("unit_price") > 0)
        & F.col("total_amount").isNotNull()
        & F.col("order_timestamp").isNotNull()
    )

    # Recalculate the total to ensure consistency
    .withColumn(
        "total_amount",
        (
            F.col("quantity") * F.col("unit_price")
        ).cast("decimal(12,2)")
    )

    # Add processing timestamp
    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

print("Cleaned order records:", silver_orders_df.count())

display(silver_orders_df.limit(10))

In [0]:
# ============================================
# 09. WRITE SILVER CUSTOMERS
# ============================================

(
    silver_customers_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_CUSTOMERS)
)

print("Silver customers table created:", SILVER_CUSTOMERS)

In [0]:
# ============================================
# 10. WRITE SILVER BOOKS
# ============================================

(
    silver_books_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_BOOKS)
)

print("Silver books table created:", SILVER_BOOKS)

In [0]:
# ============================================
# 11. WRITE SILVER ORDERS
# ============================================

(
    silver_orders_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_ORDERS)
)

print("Silver orders table created:", SILVER_ORDERS)

In [0]:
# ============================================
# 12. JOIN SILVER TABLES
# ============================================

customers_for_join = (
    spark.table(SILVER_CUSTOMERS)
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "city"
    )
)

books_for_join = (
    spark.table(SILVER_BOOKS)
    .select(
        "book_id",
        "title",
        "category",
        "price"
    )
)

orders_for_join = spark.table(SILVER_ORDERS)

silver_order_details_df = (
    orders_for_join.alias("o")

    .join(
        customers_for_join.alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"),
        "left"
    )

    .join(
        books_for_join.alias("b"),
        F.col("o.book_id") == F.col("b.book_id"),
        "left"
    )

    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("c.first_name"),
        F.col("c.last_name"),
        F.col("c.email"),
        F.col("c.city"),
        F.col("o.book_id"),
        F.col("b.title").alias("book_title"),
        F.col("b.category").alias("book_category"),
        F.col("o.quantity"),
        F.col("o.unit_price"),
        F.col("o.total_amount"),
        F.col("o.order_status"),
        F.col("o.order_timestamp")
    )

    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

print("Order detail records:", silver_order_details_df.count())

display(silver_order_details_df.limit(10))

In [0]:
# ============================================
# 13. WRITE SILVER ORDER DETAILS
# ============================================

(
    silver_order_details_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_ORDER_DETAILS)
)

print(
    "Silver order details table created:",
    SILVER_ORDER_DETAILS
)

In [0]:
# ============================================
# 14. VERIFY SILVER TABLES
# ============================================

print("SILVER CUSTOMERS")
display(
    spark.sql(f"""
        SELECT *
        FROM {SILVER_CUSTOMERS}
        LIMIT 10
    """)
)

print("SILVER BOOKS")
display(
    spark.sql(f"""
        SELECT *
        FROM {SILVER_BOOKS}
        LIMIT 10
    """)
)

print("SILVER ORDERS")
display(
    spark.sql(f"""
        SELECT *
        FROM {SILVER_ORDERS}
        LIMIT 10
    """)
)

print("SILVER ORDER DETAILS")
display(
    spark.sql(f"""
        SELECT *
        FROM {SILVER_ORDER_DETAILS}
        LIMIT 10
    """)
)

In [0]:
# ============================================
# 15. BRONZE VS SILVER COUNTS
# ============================================

count_checks = [
    ("Customers", BRONZE_CUSTOMERS, SILVER_CUSTOMERS),
    ("Books", BRONZE_BOOKS, SILVER_BOOKS),
    ("Orders", BRONZE_ORDERS, SILVER_ORDERS)
]

for name, bronze_table, silver_table in count_checks:

    bronze_count = spark.table(bronze_table).count()
    silver_count = spark.table(silver_table).count()

    print(f"{name}")
    print(f"  Bronze count: {bronze_count}")
    print(f"  Silver count: {silver_count}")
    print(f"  Removed or filtered: {bronze_count - silver_count}")
    print("-" * 50)

In [0]:
# ============================================
# 16. SILVER DATA QUALITY CHECKS
# ============================================

print("Duplicate customer IDs:")
display(
    spark.sql(f"""
        SELECT customer_id, COUNT(*) AS record_count
        FROM {SILVER_CUSTOMERS}
        GROUP BY customer_id
        HAVING COUNT(*) > 1
    """)
)

print("Invalid book prices:")
display(
    spark.sql(f"""
        SELECT *
        FROM {SILVER_BOOKS}
        WHERE price IS NULL OR price <= 0
    """)
)

print("Invalid order quantities:")
display(
    spark.sql(f"""
        SELECT *
        FROM {SILVER_ORDERS}
        WHERE quantity IS NULL OR quantity <= 0
    """)
)

print("Orders without matching customers:")
display(
    spark.sql(f"""
        SELECT o.order_id, o.customer_id
        FROM {SILVER_ORDERS} o
        LEFT ANTI JOIN {SILVER_CUSTOMERS} c
            ON o.customer_id = c.customer_id
    """)
)

print("Orders without matching books:")
display(
    spark.sql(f"""
        SELECT o.order_id, o.book_id
        FROM {SILVER_ORDERS} o
        LEFT ANTI JOIN {SILVER_BOOKS} b
            ON o.book_id = b.book_id
    """)
)

In [0]:
# ============================================
# 17. SILVER TABLE SUMMARY
# ============================================

summary = [
    (SILVER_CUSTOMERS, "Customers"),
    (SILVER_BOOKS, "Books"),
    (SILVER_ORDERS, "Orders"),
    (SILVER_ORDER_DETAILS, "Order Details")
]

for table_name, description in summary:

    count = spark.table(table_name).count()

    print(f"{description}: {table_name}")
    print(f"Record count: {count}")
    print("-" * 60)